# Data cleaning practice: diabetes readmission

**Your assignment:** audit the historical hospital encounters before choosing a model. Write the queries yourself, run each cell, and explain what its output means. We will add cleaning and preprocessing code as you work through the decisions.

The source is the [UCI Diabetes 130-US Hospitals dataset](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008). One row is one encounter. The positive outcome for this project is a recorded `readmitted == "<30"` at discharge.

**Notebook rule:** `raw` is the untouched DataFrame loaded from the CSV. Never filter or overwrite it. Later, create a separate `clean` DataFrame. Run cells from top to bottom after reconnecting to a runtime.

## 0. Load and verify the source

The path, file size, and shape are printed together so you can see which file and DataFrame you are working with. `keep_default_na=False` preserves literal categories such as `None`; we will inspect missing markers explicitly.

**Colab upload check:** the complete CSV in this project is 19,159,383 bytes and reads as 101,766 rows Ã— 50 columns. A prior run read only 12,582,912 bytes while the upload was incomplete. Wait for the upload to finish, then rerun this cell.

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path.cwd() / "diabetic_data.csv",
    Path.cwd().parent / "diabetic_data.csv",
]
csv_path = next((path for path in candidates if path.is_file()), None)
if csv_path is None:
    raise FileNotFoundError("Place diabetic_data.csv beside this notebook and run again.")

expected_bytes = 19_159_383
actual_bytes = csv_path.stat().st_size
if actual_bytes != expected_bytes:
    raise ValueError(
        f"CSV has {actual_bytes:,} bytes; expected {expected_bytes:,}. "
        "Wait for the upload to finish, then rerun this cell."
    )

raw = pd.read_csv(csv_path, keep_default_na=False, low_memory=False)
if raw.shape != (101_766, 50):
    raise ValueError(f"Expected 101,766 rows and 50 columns; found {raw.shape}.")

print("CSV:", csv_path.resolve())
print("File size:", actual_bytes, "bytes")
print("Shape:", raw.shape)
print("Unique encounter IDs:", raw["encounter_id"].nunique())
print("Unique patients:", raw["patient_nbr"].nunique())

In [ ]:
fresh = pd.read_csv(csv_path, keep_default_na=False)


## 1. Check row identity

An encounter ID should identify one row. The same patient can have several encounters. A duplicate `patient_nbr` is therefore not automatically a duplicate record.

In [ ]:
print("Repeated encounter IDs:", fresh["encounter_id"].duplicated().sum())
print("Exact duplicate rows:", fresh.duplicated().sum())

The query below counts patients with more than one encounter. Repeated patients are valid records; later, the same patient must stay in one train/test group.

In [ ]:
# TODO: Count patients with more than one encounter.
encounters_per_patient = fresh["patient_nbr"].value_counts() # number of encounters for each patient
count_patients_more = (encounters_per_patient > 1).sum()
print(count_patients_more)

## 2. Audit explicit missing markers

The CSV uses `?` in several columns. Count how many rows have `?` in each column, then express the count as a percentage of all rows. A value of zero here means only that the column has no literal `?`; we will check other unknown categories separately.

In [ ]:
if raw.shape != (101_766, 50):
    raise ValueError("raw is not the full source; rerun the load cell.")

question_mark_counts = (raw == "?").sum()
question_mark_percent = (
    question_mark_counts / len(raw) * 100
).sort_values(ascending=False)

print(question_mark_percent[question_mark_percent > 0].to_string())

In [ ]:
print(fresh.head(2))

## 3. Inspect values before choosing cleaning rules

The query below shows every distinct value and its count for `age`, `weight`, and `gender`. The full-file check above catches incomplete uploads. Decide whether these fields are numeric measurements or categories, and whether an unknown value calls for removing an entire row. Do not clean anything yet.

In [ ]:
# TODO: Show value counts for age, weight, and gender.
for column in ["age", "weight", "gender"]:
  print(f"\n{column}")
  print(fresh[column].value_counts(dropna=False))

## Decisions to explain in an interview

- Why can `patient_nbr` repeat while `encounter_id` should not?
- Why do we count missing markers before dropping or imputing anything?
- What does `None` mean in the lab-result fields, and how is it different from `?`?
- Why must we split by patient before fitting imputation and encoding?

**Later stages:** inspect coded admission/discharge fields and numeric ranges; define the eligible cohort and binary target; compare readmission counts and rates across groups; split by patient; fit preprocessing on training data only. We will add these stages as we reach them.